# EDA: LLM Jailbreak Detector Dataset

수집된 원시 데이터(`data/raw/dataset_raw.csv`)에 대한 탐색적 데이터 분석입니다.  
핵심 관심사: **길이 편향(length bias)** — jailbreak vs. normal 프롬프트의 텍스트 길이 차이가  
모델이 내용이 아닌 길이만 학습하는 지름길(shortcut)이 될 위험을 진단합니다.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from collections import Counter
from pathlib import Path

sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
PLOT_DIR = Path('../results/plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Load raw dataset
df = pd.read_csv('../data/raw/dataset_raw.csv')
df['char_len']   = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()
df['label_name'] = df['label'].map({0: 'Normal', 1: 'Jailbreak'})

jb = df[df['label'] == 1]
nm = df[df['label'] == 0]

print(f"Total : {len(df):,}")
print(f"Jailbreak (1): {len(jb):,}  |  Normal (0): {len(nm):,}")

---
## 1. 기본 통계

클래스별 텍스트 길이의 핵심 수치를 먼저 확인합니다.

In [ ]:
stats = df.groupby('label_name')['char_len'].describe()[['min','25%','50%','mean','75%','max']]
stats.columns = ['Min', 'Q1', 'Median', 'Mean', 'Q3', 'Max']
stats = stats.round(0).astype(int)
print("=== Char-length statistics ===")
print(stats.to_string())

ratio = jb['char_len'].mean() / nm['char_len'].mean()
print(f"\nMean ratio (Jailbreak / Normal): {ratio:.1f}x")

---
## 2. 문자 길이 분포 히스토그램

두 클래스의 텍스트 길이 분포를 겹쳐서 시각화합니다.  
극단적으로 긴 outlier가 시각화를 방해하므로 99 percentile 이하로 x축을 제한합니다.

In [ ]:
x_cap = int(df['char_len'].quantile(0.99))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlapping histogram
ax = axes[0]
ax.hist(nm['char_len'].clip(upper=x_cap), bins=60, alpha=0.6, color='steelblue',  label='Normal')
ax.hist(jb['char_len'].clip(upper=x_cap), bins=60, alpha=0.6, color='tomato',     label='Jailbreak')
ax.axvline(nm['char_len'].mean(), color='steelblue', linestyle='--', linewidth=1.5)
ax.axvline(jb['char_len'].mean(), color='tomato',    linestyle='--', linewidth=1.5)
ax.set_xlabel('Character length (capped at 99th pct)')
ax.set_ylabel('Count')
ax.set_title('Character Length Distribution')
ax.legend()

# Right: log-scale to see Normal distribution clearly
ax2 = axes[1]
ax2.hist(nm['char_len'].clip(upper=x_cap), bins=60, alpha=0.6, color='steelblue', label='Normal')
ax2.hist(jb['char_len'].clip(upper=x_cap), bins=60, alpha=0.6, color='tomato',    label='Jailbreak')
ax2.set_yscale('log')
ax2.set_xlabel('Character length (capped at 99th pct)')
ax2.set_ylabel('Count (log scale)')
ax2.set_title('Character Length Distribution (Log Scale)')
ax2.legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / 'char_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: results/plots/char_length_dist.png")

---
## 3. 단어 수 분포

문자 수 대신 단어 수로 봐도 같은 패턴이 나타나는지 확인합니다.

In [ ]:
wc_cap = int(df['word_count'].quantile(0.99))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(nm['word_count'].clip(upper=wc_cap), bins=50, alpha=0.6, color='steelblue', label='Normal')
ax.hist(jb['word_count'].clip(upper=wc_cap), bins=50, alpha=0.6, color='tomato',    label='Jailbreak')
ax.axvline(nm['word_count'].mean(), color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Normal mean={nm["word_count"].mean():.0f}')
ax.axvline(jb['word_count'].mean(), color='tomato',    linestyle='--', linewidth=1.5,
           label=f'Jailbreak mean={jb["word_count"].mean():.0f}')
ax.set_xlabel('Word count (capped at 99th pct)')
ax.set_ylabel('Count')
ax.set_title('Word Count Distribution')
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'word_count_dist.png', dpi=150, bbox_inches='tight')
plt.show()

wc_ratio = jb['word_count'].mean() / nm['word_count'].mean()
print(f"Word count ratio (JB / Normal): {wc_ratio:.1f}x")

---
## 4. 길이 편향 시각화 (Box Plot)

클래스별 분포 차이를 박스플롯으로 직관적으로 보여줍니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Char length boxplot
sns.boxplot(
    data=df[df['char_len'] <= x_cap],
    x='label_name', y='char_len',
    palette={'Normal': 'steelblue', 'Jailbreak': 'tomato'},
    ax=axes[0]
)
axes[0].set_title('Character Length by Class')
axes[0].set_xlabel('')
axes[0].set_ylabel('Character length')

# Word count boxplot
sns.boxplot(
    data=df[df['word_count'] <= wc_cap],
    x='label_name', y='word_count',
    palette={'Normal': 'steelblue', 'Jailbreak': 'tomato'},
    ax=axes[1]
)
axes[1].set_title('Word Count by Class')
axes[1].set_xlabel('')
axes[1].set_ylabel('Word count')

plt.suptitle('Length Bias Analysis: Jailbreak vs. Normal Prompts', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'bias_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/plots/bias_boxplot.png")

---
## 5. 키워드 빈도 분석

각 클래스에서 가장 자주 등장하는 단어를 확인합니다.  
관사·전치사 등 stopword는 제외합니다.

In [ ]:
STOPWORDS = {
    'the','a','an','is','in','to','of','and','or','for','with','on','at',
    'it','its','this','that','as','are','was','be','by','have','has','had',
    'i','you','we','he','she','they','my','your','our','their','from','not',
    'can','will','do','does','did','what','how','when','where','which','who',
    'if','but','so','no','all','any','more','about','me','him','her','them'
}

def top_words(series: pd.Series, n: int = 20) -> pd.DataFrame:
    tokens = []
    for text in series:
        for w in str(text).lower().split():
            w = w.strip('.,!?;:\'"()[]{}')
            if w and w not in STOPWORDS and len(w) > 2:
                tokens.append(w)
    cnt = Counter(tokens)
    return pd.DataFrame(cnt.most_common(n), columns=['word', 'count'])

top_jb = top_words(jb['text'])
top_nm = top_words(nm['text'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=top_jb, x='count', y='word', palette='Reds_r', ax=axes[0])
axes[0].set_title('Top 20 Words — Jailbreak')
axes[0].set_xlabel('Frequency')

sns.barplot(data=top_nm, x='count', y='word', palette='Blues_r', ax=axes[1])
axes[1].set_title('Top 20 Words — Normal')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'keyword_freq.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/plots/keyword_freq.png")

In [ ]:
# Side-by-side frequency table
top_jb_r = top_jb.rename(columns={'word':'JB Word','count':'JB Count'})
top_nm_r = top_nm.rename(columns={'word':'NM Word','count':'NM Count'})
combined = pd.concat([top_jb_r.reset_index(drop=True), top_nm_r.reset_index(drop=True)], axis=1)
print(combined.to_string(index=False))

---
## 6. 길이 분포 누적 곡선 (CDF)

BERT `max_length=128`에 해당하는 토큰 수 기준으로 얼마나 많은 샘플이 절단되는지 추정합니다.  
대략적으로 **4~5 chars ≈ 1 token** 을 사용합니다.

In [ ]:
# Approximate token count (1 token ≈ 4.5 chars for English)
df['approx_tokens'] = (df['char_len'] / 4.5).round().astype(int)
jb2 = df[df['label'] == 1]
nm2 = df[df['label'] == 0]

fig, ax = plt.subplots(figsize=(10, 5))

for subset, color, label in [(nm2, 'steelblue', 'Normal'), (jb2, 'tomato', 'Jailbreak')]:
    sorted_vals = np.sort(subset['approx_tokens'].clip(upper=300))
    cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    ax.plot(sorted_vals, cdf, color=color, label=label, linewidth=2)

ax.axvline(128, color='black', linestyle='--', linewidth=1.5, label='BERT max_length=128')
ax.set_xlabel('Approximate token count')
ax.set_ylabel('Cumulative fraction')
ax.set_title('CDF of Approximate Token Count — BERT truncation boundary')
ax.legend()
ax.set_xlim(0, 300)

# Annotate what fraction is truncated
jb_trunc = (jb2['approx_tokens'] > 128).mean()
nm_trunc = (nm2['approx_tokens'] > 128).mean()
ax.text(135, 0.55, f'Truncated\nJB: {jb_trunc:.0%}\nNM: {nm_trunc:.0%}', fontsize=10)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'token_cdf.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Estimated truncation at max_length=128 — Jailbreak: {jb_trunc:.1%}, Normal: {nm_trunc:.1%}")

---
## 7. 분석 요약 및 전처리 전략

### 핵심 발견

| 지표 | Jailbreak | Normal | 비율 |
|------|-----------|--------|------|
| 평균 문자 길이 | ~1,879 | ~60 | **~30x** |
| 평균 단어 수 | ~350 | ~12 | **~29x** |
| 예상 토큰 수 | ~417 | ~13 | **~32x** |

### 위험: 길이 지름길(length shortcut)

- BERT가 학습 중 "입력이 길면 jailbreak" 패턴을 학습할 가능성이 높음
- 이 경우 모델은 실제 jailbreak 의도가 아닌 텍스트 길이만으로 분류
- 짧게 작성된 jailbreak에 대한 일반화 실패

### 전처리 전략 (src/dataset.py)

1. **전략 (b)** — Normal 재수집: Alpaca의 `instruction + input` 연결,  
   dolly-15k (`instruction + context`) 추가  
   → 정상 프롬프트 평균 길이 ~60 → ~200+ chars로 향상

2. **전략 (a)** — 최소 길이 필터: 50 chars 미만 정상 프롬프트 제거  
   → 극단적으로 짧은 정상 샘플 제거

3. **전략 (c)** — `max_length=128` 토큰 상한  
   → jailbreak 프롬프트의 과도한 길이를 동등하게 절단  
   → 길이 차이가 모델 입력에 미치는 영향 최소화

세 전략 조합으로 토큰 수준 비율을 **~32x → ~3x 이하**로 감소 목표